# Notebook for Milestone - 1

In [2]:
import os
import sys
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
from src.preprocess import clean_text, tokenize, remove_stopwords, get_combined_texts
from src.utils import map_at_3, load_config, set_seed

warnings.filterwarnings('ignore')
load_dotenv('../.env')
set_seed(42)

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
TRAIN_PATH  = '../data/raw/train.csv'



In [3]:
df = pd.read_csv(TRAIN_PATH)
print(f'Shape: {df.shape}')
df.head(3)

Shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


## Q1

In [4]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Answer Distribution ===')
print(df['answer'].value_counts())

=== Dataset Info ===
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2000 non-null   int64
 1   prompt  2000 non-null   str  
 2   A       2000 non-null   str  
 3   B       2000 non-null   str  
 4   C       2000 non-null   str  
 5   D       2000 non-null   str  
 6   E       2000 non-null   str  
 7   answer  2000 non-null   str  
dtypes: int64(1), str(7)
memory usage: 1.9 MB

=== Missing Values ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== Answer Distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [5]:
freq = df['answer'].value_counts()
most_frequent  = freq.index[0]
least_frequent = freq.index[-1]

q1_answer = freq[most_frequent] + freq[least_frequent]

print(f'Answer frequencies:\n{freq}')
print(f'\nMost frequent  : {most_frequent} ({freq[most_frequent]})')
print(f'Least frequent : {least_frequent} ({freq[least_frequent]})')
print(f'\nA1: {q1_answer}')

Answer frequencies:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent  : B (490)
Least frequent : E (324)

A1: 814


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Answer distribution
freq.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Correct Answer Distribution')
axes[0].set_xlabel('Answer')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Prompt length distribution
df['prompt_len'] = df['prompt'].apply(lambda x: len(str(x).split()))
axes[1].hist(df['prompt_len'], bins=30, color='coral', edgecolor='black')
axes[1].set_title('Prompt Length Distribution (words)')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()
print(f'Prompt length stats:\n{df["prompt_len"].describe()}')